# 🔐 The Secrets of Python
### Self-Learning Notebook — Tricks, Gotchas & Hidden Gems

---

## 📋 Table of Contents
1. [`__name__` — Module vs Standalone Execution](#1-__name__--module-vs-standalone-execution)
2. [`f''` vs `r''` String Prefixes](#2-f-vs-r-string-prefixes)
3. [`try / except / else / finally`](#3-try--except--else--finally)
4. [List Operations & Tricks](#4-list-operations--tricks)
5. [2D Lists](#5-2d-lists)
6. [Tuple vs Integer — The Single-Element Trap](#6-tuple-vs-integer--the-single-element-trap)
7. [Bytes & Type Conversion Gotchas](#7-bytes--type-conversion-gotchas)
8. [`zip()` — Pairing & Unzipping](#8-zip--pairing--unzipping)
9. [Hashability — Mutable vs Immutable](#9-hashability--mutable-vs-immutable)
10. [Dictionary Comprehensions](#10-dictionary-comprehensions)
11. [Line Continuation with `\`](#11-line-continuation-with-)
12. [The `datetime` Module](#12-the-datetime-module)
13. [Function Secrets](#13-function-secrets)
14. [Miscellaneous Gotchas](#14-miscellaneous-gotchas)

---

## 1. `__name__` — Module vs Standalone Execution

Every Python file has a built-in variable called `__name__`. Its value changes depending on **how** the file is run:

| How the file is run | Value of `__name__` |
|---|---|
| Executed directly (`python file.py`) | `'__main__'` |
| Imported by another file (`import file`) | `'file'` (the filename without `.py`) |

This is why you often see the guard:
```python
if __name__ == '__main__':
    ...
```
Code inside that block runs **only** when the file is executed directly — it is **skipped** when the file is imported as a module.

> 💡 Functions also have a `__name__` attribute — it stores the **function's own name** as a string (see Section 13).

In [ ]:
def greet():
    print('Hello from greet()!')

# This block only runs when THIS file is executed directly.
# If another file imports this one, the block is skipped.
if __name__ == '__main__':
    greet()

# Show the current value of __name__ in this notebook:
print('Current __name__:', __name__)
# In a notebook / direct run → '__main__'
# When imported → the module's filename

## 2. `f''` vs `r''` String Prefixes

Python has several string prefixes. Two of the most important are `f` and `r`:

| Prefix | Name | How it handles `\` (backslash) |
|---|---|---|
| `f'...'` | **f-string** (formatted string) | Processes escape sequences normally (`\n` = newline, `\t` = tab) |
| `r'...'` | **raw string** | Treats `\` as a **literal** backslash character — no escape processing |

> 💡 Raw strings are especially useful for **file paths** on Windows (`r'C:\Users\name'`) and **regular expressions** where backslashes are common.

In [ ]:
# f-string: \n is treated as a real newline character
print(f'text one\ntext two')
# Output:
# text one
# text two

print()

# r-string: \n is treated as the two characters \ and n (no newline)
print(r'text one\ntext two')
# Output:
# text one\ntext two

print()

# Practical example: Windows file path
path_normal = 'C:\\Users\\islam\\Documents'   # need double \\
path_raw    = r'C:\Users\islam\Documents'      # clean and readable
print('Normal:', path_normal)
print('Raw   :', path_raw)

## 3. `try / except / else / finally`

Python's exception handling has **four** clauses — most beginners only know the first two:

```python
try:
    # code that might raise an exception
except SomeError as e:
    # runs ONLY if the exception occurred
else:
    # runs ONLY if NO exception occurred
finally:
    # ALWAYS runs — exception or not
```

| Clause | When does it run? |
|---|---|
| `try` | Always attempted |
| `except` | Only when the specified error is raised |
| `else` | Only when the `try` block succeeds (no exception) |
| `finally` | **Always** — success or failure, it runs no matter what |

> 💡 `finally` is typically used for **cleanup** — closing files, releasing resources, etc.

In [ ]:
# --- Scenario 1: Valid input (integer entered) ---
# Try changing the value of user_input to test both paths

user_input = '42'   # ← change to 'hello' to trigger the except branch

try:
    var = int(user_input)
except ValueError as e:
    print('except  → Error caught:', e)
else:
    print('else    → No error! var =', var)
finally:
    print('finally → This ALWAYS runs')

In [ ]:
# --- Scenario 2: Invalid input (non-integer) ---

user_input = 'hello'

try:
    var = int(user_input)
except ValueError as e:
    print('except  → Error caught:', e)
else:
    print('else    → No error! var =', var)  # this line is SKIPPED
finally:
    print('finally → This ALWAYS runs')      # this line still runs

## 4. List Operations & Tricks

Lists support several arithmetic-like operators:

| Operation | Example | Result |
|---|---|---|
| Repetition (list level) | `['x'] * 3` | `['x', 'x', 'x']` |
| Repetition (element level) | `['x' * 3]` | `['xxx']` |
| Concatenation | `[] + 2 * [1,2,3]` | `[1, 2, 3, 1, 2, 3]` |

> ⚠️ The position of `*` matters enormously:
> - `['x'] * 3` → repeats the **list** (three separate `'x'` elements)
> - `['x' * 3]` → repeats the **string** first, then wraps it in a list

In [ ]:
# --- List repetition ---
List  = ['x'] * 3    # repeats the list element → 3 separate items
ListX = ['x' * 3]    # repeats the string first → one combined item

print('List  :', List)    # ['x', 'x', 'x']
print('ListX :', ListX)   # ['xxx']

print()

# --- Concatenation and multiplication combined ---
base  = [1, 2, 3]
ListY = [] + 2 * base
print('[] + 2 * [1,2,3] :', ListY)   # [1, 2, 3, 1, 2, 3]

# Same result using the + operator on two list copies:
print('base + base       :', base + base)  # [1, 2, 3, 1, 2, 3]

## 5. 2D Lists

A **2D list** is a list whose elements are themselves lists — think of it as a table with rows and columns.

```
data_list = [ [name1, name2, ...],       ← row 0: names
              [id1,   id2,   ...],       ← row 1: IDs
              [ph1,   ph2,   ...] ]      ← row 2: phone numbers
```

**Key rule:**
> The **number of dimensions** of a list = the **number of nested `for` loops** needed to print all its contents individually.

| Dimensions | Loops needed |
|---|---|
| 1D list | 1 loop |
| 2D list | 2 loops |
| 3D list | 3 loops |

> 💡 For storing structured data (names + IDs + phones), **dictionaries** give better performance and clarity than 2D lists.

In [ ]:
# --- Building a 2D list from user data ---
# (Using preset data here instead of input() for notebook compatibility)

data_list = [[], [], []]   # row 0: names | row 1: IDs | row 2: phone numbers

records = [
    ('Islam',  'ID001', '059-000-0001'),
    ('Anas',   'ID002', '059-000-0002'),
    ('Yousef', 'ID003', '059-000-0003'),
]

for name, ID, phone in records:
    data_list[0].append(name)
    data_list[1].append(ID)
    data_list[2].append(phone)

print('Names  :', data_list[0])
print('IDs    :', data_list[1])
print('Phones :', data_list[2])

print()

# --- Printing all contents with 2 nested loops ---
labels = ['Name', 'ID', 'Phone']
for row_index, row in enumerate(data_list):       # outer loop: rows
    print(f'{labels[row_index]}:')
    for item in row:                               # inner loop: items in each row
        print(f'  {item}')

## 6. Tuple vs Integer — The Single-Element Trap

When creating a tuple with only **one element**, you **must** include a trailing comma:

```python
var = (1,)   # ✅ This is a TUPLE   — the comma is what makes it a tuple
var = (1)    # ❌ This is an INTEGER — parentheses here are just grouping
```

Python uses parentheses for **grouping** in many places (math, function calls, etc.), so without the comma, `(1)` is simply the integer `1` wrapped in grouping parentheses.

> 💡 Remember: it's the **comma** that creates a tuple, not the parentheses.

In [ ]:
a = (1,)   # tuple with one element
b = (1)    # just the integer 1
c = 1,     # also a tuple! (parentheses are actually optional)

print('(1,)  →', type(a).__name__, '=', a)   # tuple = (1,)
print('(1)   →', type(b).__name__, '=', b)   # int   = 1
print('1,    →', type(c).__name__, '=', c)   # tuple = (1,)

# Contrast with multi-element tuples (commas make it clear):
d = (1, 2, 3)
e = 1, 2, 3    # same thing — parentheses are optional for multi-element tuples
print('\n(1,2,3) →', type(d).__name__, '=', d)
print('1,2,3   →', type(e).__name__, '=', e)

## 7. Bytes & Type Conversion Gotchas

### Bytes Literals (`b'...'`)
A **bytes literal** is a sequence of raw bytes. When you convert it to a list using `list()`, you get the **ASCII/Unicode code** of each character:

```python
list(b'hi')   # → [104, 105]   (ASCII codes for 'h' and 'i')
```

### Type Conversion Rules

| Conversion | Works? | Notes |
|---|---|---|
| `int('99.99')` | ❌ Error | Can't convert a decimal string directly to int |
| `int(float('99.99'))` | ✅ | Convert to float first, then to int |
| `tuple(1000)` | ❌ Error | Integers are not iterable |
| `tuple('islam')` | ✅ | Strings are iterable → tuple of characters |

In [ ]:
# --- Bytes literal → list of ASCII codes ---
byte_string = b'a text'
print('bytes literal :', byte_string)
print('list(b"a text"):', list(byte_string))   # [97, 32, 116, 101, 120, 116]

# Verify: ASCII code 97 = 'a'
print('chr(97)       :', chr(97))    # 'a'
print('ord("a")      :', ord('a'))   # 97

In [ ]:
# --- Type conversion gotchas ---

f = '99.99'

# int('99.99') → ERROR: string contains a decimal point
try:
    print(int(f))
except ValueError as e:
    print('int("99.99") →', e)

# Correct: float first, then int (truncates the decimal part)
print('int(float("99.99")) →', int(float(f)))   # 99

print()

# tuple(integer) → ERROR: int is not iterable
try:
    print(tuple(1000))
except TypeError as e:
    print('tuple(1000)   →', e)

# tuple(string) → works! strings are iterable
print('tuple("islam") →', tuple('islam'))   # ('i', 's', 'l', 'a', 'm')

## 8. `zip()` — Pairing & Unzipping

`zip()` combines multiple iterables **element by element** into tuples:

```python
zip([1,2,3], [4,5,6])  →  (1,4), (2,5), (3,6)
```

**Important rules:**
- `zip()` stops at the **shortest** list — extra elements in longer lists are ignored
- `zip(*zipped)` is the **inverse** operation (unzip) — it transposes rows and columns

| Operation | Description |
|---|---|
| `zip(a, b)` | Pairs elements from `a` and `b` together |
| `zip(a, [x**2 for x in a])` | Pairs each element with its square |
| `zip(*zip(a, b))` | Unzips — separates the pairs back into original groups |

In [ ]:
# --- Basic zip: pair each number with its square ---
nums = [1, 2, 3]

print('--- zip(nums, squares) ---')
for pair in zip(nums, [n**2 for n in nums]):
    print(pair)
# (1, 1)
# (2, 4)
# (3, 9)

In [ ]:
# --- zip(*zip(...)): unzipping (transpose) ---
# zip(nums, squares)   gives: (1,1), (2,4), (3,9)   ← rows of pairs
# zip(*zip(...))       transposes: (1,2,3), (1,4,9)  ← columns separated

nums = [1, 2, 3]
print('--- zip(*zip(nums, squares)) ---')
for group in zip(*zip(nums, [n**2 for n in nums])):
    print(group)
# (1, 2, 3)    ← all original numbers
# (1, 4, 9)    ← all squares

In [ ]:
# --- zip stops at the SHORTEST list ---
long_list  = [1, 2, 3]
short_list = [10, 20]     # only 2 elements

print('zip stops at shortest:')
for pair in zip(long_list, short_list):
    print(pair)
# (1, 10)
# (2, 20)
# ← 3 is dropped because short_list ran out

## 9. Hashability — Mutable vs Immutable

In Python, an object is **hashable** if it has a fixed hash value that never changes during its lifetime.

**The rule:**
> Every Python built-in type that is **immutable** is also **hashable**.
> **Mutable** types — `list`, `dict`, `set` — are **not hashable**.

| Type | Mutable? | Hashable? | Can be a dict key? |
|---|---|---|---|
| `str` | ❌ No | ✅ Yes | ✅ Yes |
| `int`, `float` | ❌ No | ✅ Yes | ✅ Yes |
| `tuple` | ❌ No | ✅ Yes | ✅ Yes |
| `list` | ✅ Yes | ❌ No | ❌ No |
| `dict` | ✅ Yes | ❌ No | ❌ No |
| `set` | ✅ Yes | ❌ No | ❌ No |

- `__hash__()` returns the hash integer of an object
- `__eq__(other)` returns `True` if the two objects have the **same value**

In [ ]:
# --- Strings are hashable ---
var  = 'islam ouda'
vare = 'islam ouda'

print('hash(var)        :', var.__hash__())    # some integer
print('var == vare      :', var.__eq__(vare))  # True (same value)
print('var == "other"   :', var.__eq__('other'))  # False

# Strings can be dict keys:
d = {'islam ouda': 'author', 'python': 'language'}
print('\ndict lookup:', d['islam ouda'])

print()

# --- Lists are NOT hashable ---
my_list = [1, 2, 3]
try:
    hash(my_list)   # will raise TypeError
except TypeError as e:
    print('hash(list) →', e)

# But a TUPLE (immutable) is hashable:
my_tuple = (1, 2, 3)
print('hash(tuple) →', hash(my_tuple))   # works fine

## 10. Dictionary Comprehensions

Just like list comprehensions create lists, **dictionary comprehensions** create dictionaries:

```python
{key_expr: value_expr  for item in iterable  if condition}
```

You can also combine them with `reversed()` to control the insertion order:

```python
{str(n): None for n in range(5)}           # keys: '0','1','2','3','4'
{str(n): None for n in reversed(range(5))} # keys: '4','3','2','1','0'
```

> 💡 In Python 3.7+, dictionaries maintain **insertion order** — so the order you build them matters.

In [ ]:
# --- Basic dictionary comprehension ---
squares = {str(n): n**2 for n in range(6)}
print('Squares dict :', squares)
# {'0': 0, '1': 1, '2': 4, '3': 9, '4': 16, '5': 25}

# --- Using None as a placeholder value (just for the keys) ---
forward = {str(n): None for n in range(5)}
backward = {str(n): None for n in reversed(range(5))}

print('\nForward keys :', list(forward.keys()))   # ['0','1','2','3','4']
print('Backward keys:', list(backward.keys()))   # ['4','3','2','1','0']

# --- With a condition ---
even_squares = {n: n**2 for n in range(10) if n % 2 == 0}
print('\nEven squares :', even_squares)

In [ ]:
# --- Named placeholders in format() ---
# You can use {name} syntax and pass keyword arguments to format()

print('my name is {islam}'.format(islam='ali'))
# Output: my name is ali

print('Hello, {first} {last}!'.format(first='Islam', last='Ouda'))
# Output: Hello, Islam Ouda!

# This is equivalent to using f-strings:
first, last = 'Islam', 'Ouda'
print(f'Hello, {first} {last}!')

## 11. Line Continuation with `\`

When a single statement is very long, you can split it across multiple lines using a **backslash** `\` at the end of a line:

```python
result = first_part + \
         second_part
```

Python reads this as one continuous line. This is useful for readability when writing long expressions.

> 💡 **Alternative:** Inside `()`, `[]`, or `{}`, Python allows implicit line continuation — no `\` needed.
> ```python
> result = (
>     first_part +
>     second_part
> )
> ```

In [ ]:
# --- Line continuation with \ ---
total = sum(x**2 \
            for x in range(5))
print('Sum of squares (0²+1²+2²+3²+4²):', total)   # 0+1+4+9+16 = 30

# --- Implicit continuation inside parentheses (preferred style) ---
total2 = sum(
    x**2
    for x in range(5)
)
print('Same result                      :', total2)

# --- Another example: long condition ---
x = 7
is_valid = (x > 0 and
            x < 10 and
            x != 5)
print('is_valid (0 < x < 10, x≠5)      :', is_valid)

## 12. The `datetime` Module

Python's built-in `datetime` module lets you work with dates and times. The most common entry point is `datetime.datetime.today()`, which returns the current local date and time.

You can format the output using **format codes** inside an f-string:

| Code | Meaning | Example |
|---|---|---|
| `%d` | Day of the month (zero-padded) | `05` |
| `%B` | Full month name | `January` |
| `%b` | Abbreviated month name | `Jan` |
| `%Y` | 4-digit year | `2025` |
| `%y` | 2-digit year | `25` |
| `%H` | Hour (24-hour clock) | `14` |
| `%M` | Minute | `30` |
| `%S` | Second | `45` |

In [ ]:
import datetime

today = datetime.datetime.today()
print('Raw datetime object :', today)

# Format using f-string with datetime codes
print('Formatted date      :', f'{today:%d %B, %Y}')     # e.g. 05 January, 2025
print('Abbreviated month   :', f'{today:%d %b %Y}')      # e.g. 05 Jan 2025
print('Time only           :', f'{today:%H:%M:%S}')      # e.g. 14:30:45
print('Full                :', f'{today:%d %B %Y — %H:%M}')

## 13. Function Secrets

### `__name__` attribute of functions
Every function has a `__name__` attribute that stores its name as a string. This can be used for introspection:
```python
def my_func(): pass
my_func.__name__   # → 'my_func'
```

### The `main()` convention
In Python it is **recommended** (but not required) to define a `main()` function and call it from the `if __name__ == '__main__'` guard. In Java and C++ you **must** have a `main` — in Python it is just good style.

### Order of parameters in function definitions
Python enforces a strict order when mixing different parameter types:

```python
def func(arg1, arg2, *args, kw1='a', kw2='b', **kwargs):
#         ───①───   ──②──  ──────③──────     ────④────
```

| # | Type | Example |
|---|---|---|
| ① | Regular positional arguments | `arg1, arg2` |
| ② | Variable positional (`*args`) | Collects extra positional args into a tuple |
| ③ | Default keyword arguments | `kw1='a'` |
| ④ | Variable keyword (`**kwargs`) | Collects extra keyword args into a dict |

### Unpacking a tuple/list into a function call
Use `*` to unpack an iterable into separate arguments:
```python
group = (10, 20, 30)
func(*group)   # same as func(10, 20, 30)
```

In [ ]:
# --- Function __name__ attribute ---
def func():
    print('func() was called')

print('func.__name__ :', func.__name__)   # 'func'

# Use it to call a function only if its name matches:
if func.__name__ == 'func':
    func()

In [ ]:
# --- The main() convention ---
def main():
    print('Program starting...')
    print('Running main logic here.')

if __name__ == '__main__':
    main()

In [ ]:
# --- Parameter order: positional, *args, defaults, **kwargs ---

def demo(a, b, *args, greeting='Hello', **kwargs):
    print(f'a={a}, b={b}')
    print(f'extra positional (args)  : {args}')
    print(f'greeting (default kw)    : {greeting}')
    print(f'extra keyword (kwargs)   : {kwargs}')

demo(1, 2, 3, 4, 5, greeting='Hi', name='Islam', city='Gaza')

print()

# --- Unpacking a tuple into a function call with * ---
def multiply(a, b, c):
    return a * b * c

values = (2, 3, 4)
print('multiply(*values) :', multiply(*values))   # same as multiply(2, 3, 4)

## 14. Miscellaneous Gotchas

A collection of small-but-important Python quirks and built-ins.

### Leading zeros in integer literals
You **cannot** write an integer with a leading zero in Python (unlike some other languages):
```python
x = 9    # ✅ valid
x = 09   # ❌ SyntaxError: invalid token
```

### `divmod(a, b)`
Returns a tuple `(quotient, remainder)` — equivalent to `(a // b, a % b)` in one call.

### Boolean / comparison operators are case-sensitive
String comparisons in Python are **case-sensitive** by default:
```python
'Sammy' == 'sammy'   # False
```

### Iterating with a tuple in `for` — no list needed!
You don't need to create a list variable just to loop over a few items. A tuple works fine directly in a `for` loop.

In [ ]:
# --- Leading zeros are invalid ---
variable = 9    # ✅
# variable = 09 # ❌ SyntaxError — uncomment to see the error

# Octal literals use the 0o prefix:
octal = 0o9    # ❌ also invalid (9 is not a valid octal digit)
# valid octal:
print('0o10 (octal) =', 0o10)   # = 8 in decimal

In [ ]:
# --- divmod() ---
a, b = 7, 3
result = divmod(a, b)
print('divmod(7, 3)   :', result)            # (2, 1)
print('quotient       :', result[0])         # 7 // 3 = 2
print('remainder      :', result[1])         # 7 %  3 = 1

# Useful for converting seconds to minutes and seconds:
minutes, seconds = divmod(137, 60)
print(f'\n137 seconds = {minutes}m {seconds}s')

In [ ]:
# --- Comparison operators are case-sensitive ---
Sammy = 'Sammy'
sammy = 'sammy'

print('Sammy == sammy :', Sammy == sammy)     # False
print('Sammy == Sammy :', Sammy == 'Sammy')   # True

# To compare ignoring case, use .lower():
print('case-insensitive:', Sammy.lower() == sammy.lower())  # True

In [ ]:
# --- Iterating with a tuple directly in a for loop ---
a, b, c, d = 1, 4, 9, 16

# Method 1: build a list first
list1 = [a, b, c, d]
for i in list1:
    print(i, end=' ')
print()

# Method 2: list comprehension
list2 = [i**2 for i in range(1, 5)]
for m in list2:
    print(m, end=' ')
print()

# Method 3: inline tuple — no variable needed!
for n in (a, b, c, d):
    print(n, end=' ')
print()

# All three produce the same output: 1 4 9 16
# Answer: all of the above ✅

---
## ✅ Chapter Summary

| Secret | Key Takeaway |
|---|---|
| `__name__` | `'__main__'` when run directly; module name when imported |
| `f''` vs `r''` | f-strings process `\n`; raw strings treat `\` as literal text |
| `else` in try | Runs **only** if no exception occurred |
| `finally` | **Always** runs — even after an exception |
| `['x']*3` vs `['x'*3]` | Repeats the list vs repeats the string inside |
| 2D list rule | #dimensions = #nested loops to print all items |
| Single-element tuple | Must write `(val,)` — not `(val)` |
| `list(b'text')` | Gives ASCII code integers for each character |
| `int(float('9.9'))` | Float conversion needed before int when string has decimal |
| `zip()` | Stops at shortest list; `zip(*zip(...))` unzips/transposes |
| Hashability | Immutable → hashable → usable as dict key |
| Dict comprehension | `{k: v for x in iterable}` — order is insertion order |
| `\` line continuation | Splits long lines; `()` is preferred alternative |
| `datetime` | Use `%d %B %Y` format codes in f-strings |
| `func.__name__` | Stores the function's own name as a string |
| Parameter order | positional → `*args` → default kwargs → `**kwargs` |
| `func(*tuple)` | Unpacks a tuple into separate function arguments |
| Leading zeros | `09` is a **SyntaxError** in Python |
| `divmod(a, b)` | Returns `(a // b, a % b)` as a tuple |
| Case sensitivity | `'Sammy' == 'sammy'` is `False` — use `.lower()` to compare |
| Inline tuple in `for` | `for x in (a, b, c):` — no list variable needed |

---
*Next: Files & Exceptions →*